# Predicting Temperature in London

![tower_bridge](tower_bridge.jpeg)

As the climate changes, predicting the weather becomes ever more important for businesses. Since the weather depends on a lot of different factors, you will want to run a lot of experiments to determine what the best approach is to predict the weather. In this project, you will run experiments for different regression models predicting the mean temperature, using a combination of `sklearn` and `MLflow`.

You will be working with data stored in `london_weather.csv`, which contains the following columns:
- **date** - recorded date of measurement - (**int**)
- **cloud_cover** - cloud cover measurement in oktas - (**float**)
- **sunshine** - sunshine measurement in hours (hrs) - (**float**)
- **global_radiation** - irradiance measurement in Watt per square meter (W/m2) - (**float**)
- **max_temp** - maximum temperature recorded in degrees Celsius (°C) - (**float**)
- **mean_temp** - mean temperature in degrees Celsius (°C) - (**float**)
- **min_temp** - minimum temperature recorded in degrees Celsius (°C) - (**float**)
- **precipitation** - precipitation measurement in millimeters (mm) - (**float**)
- **pressure** - pressure measurement in Pascals (Pa) - (**float**)
- **snow_depth** - snow depth measurement in centimeters (cm) - (**float**)

## 1. Import Libraries and Constants

In [ ]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.pipeline import Pipeline

TEST_SIZE = 0.2
RANDOM_STATE = 42
CV_FOLDS = 5
FIGURE_SIZE = (10, 5)
HEATMAP_SIZE = (10, 8)

## 2. Data Loading and Preprocessing

In [ ]:
def load_and_preprocess_data(filepath="london_weather.csv"):
    """Load weather data and perform initial preprocessing."""
    weather = pd.read_csv(filepath)
    
    weather['date'] = pd.to_datetime(weather['date'], format="%Y%m%d", errors='coerce')
    
    weather['year'] = weather['date'].dt.year
    weather['month'] = weather['date'].dt.month
    
    return weather

def display_data_info(weather):
    """Display basic information about the dataset."""
    print("Dataset Info:")
    print(weather.info())
    print("\nDataset Description:")
    print(weather.describe())
    print("\nMissing Values:")
    print(weather.isna().sum())
    print("\nFirst 5 rows:")
    print(weather.head())

weather = load_and_preprocess_data()
display_data_info(weather)

## 3. Exploratory Data Analysis

In [ ]:
def plot_temperature_trends(weather, figure_size=FIGURE_SIZE):
    """Plot yearly and monthly temperature trends."""
    plt.figure(figsize=figure_size)
    sns.lineplot(data=weather, x="year", y="mean_temp")
    plt.title("Yıllara Göre Ortalama Sıcaklık (London)")
    plt.xlabel("Yıl")
    plt.ylabel("Ortalama Sıcaklık (°C)")
    plt.show()

    plt.figure(figsize=figure_size)
    sns.boxplot(data=weather, x="month", y="mean_temp")
    plt.title("Aylara Göre Ortalama Sıcaklık Dağılımı")
    plt.xlabel("Ay")
    plt.ylabel("Ortalama Sıcaklık (°C)")
    plt.show()

def plot_correlation_matrix(weather, figure_size=HEATMAP_SIZE):
    """Plot correlation matrix heatmap."""
    plt.figure(figsize=figure_size)
    corr = weather.corr(numeric_only=True)
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
    plt.title("Korelasyon Matrisi")
    plt.show()

plot_temperature_trends(weather)
plot_correlation_matrix(weather)

## 4. Feature Engineering and Data Preparation

In [ ]:
def prepare_features_and_target(weather):
    """Prepare feature matrix and target variable."""
    feature_columns = [
        "cloud_cover", "sunshine", "global_radiation",
        "max_temp", "min_temp", "precipitation",
        "pressure", "snow_depth", "year", "month"
    ]
    
    X = weather[feature_columns]
    y = weather["mean_temp"]
    
    data = pd.concat([X, y], axis=1).dropna(subset=["mean_temp"])
    X = data[feature_columns]
    y = data["mean_temp"]
    
    return X, y

X, y = prepare_features_and_target(weather)
print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

## 5. Feature Selection

In [ ]:
def perform_feature_selection(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, cv_folds=CV_FOLDS):
    """Perform feature selection using SelectKBest with cross-validation."""
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_regression)),
        ("model", LinearRegression())
    ])
    
    param_grid = {
        "selector__k": range(2, X.shape[1] + 1)
    }
    
    grid = GridSearchCV(
        pipeline, param_grid, 
        scoring='neg_root_mean_squared_error', 
        cv=cv_folds
    )
    grid.fit(X_train, y_train)
    
    y_pred = grid.predict(X_test)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    best_selector = grid.best_estimator_.named_steps["selector"]
    selected_features = X.columns[best_selector.get_support()]
    
    results = {
        'best_k': grid.best_params_["selector__k"],
        'cv_rmse': -grid.best_score_,
        'test_rmse': test_rmse,
        'selected_features': selected_features.tolist(),
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test
    }
    
    return results

feature_results = perform_feature_selection(X, y)

print(f"En iyi k: {feature_results['best_k']}")
print(f"CV En iyi RMSE: {feature_results['cv_rmse']:.4f}")
print(f"Test RMSE: {feature_results['test_rmse']:.4f}")
print(f"Seçilen Özellikler: {feature_results['selected_features']}")

## 6. Model Evaluation with MLflow

In [ ]:
def evaluate_models_with_mlflow(X_train, X_test, y_train, y_test, experiment_name="weather_experiments"):
    """Evaluate multiple models using MLflow for experiment tracking."""
    mlflow.set_experiment(experiment_name)
    
    models = [
        ("LinearRegression", LinearRegression(), {}),
        ("DecisionTree_depth3", DecisionTreeRegressor(max_depth=3), {"max_depth": 3}),
        ("DecisionTree_depth5", DecisionTreeRegressor(max_depth=5), {"max_depth": 5}),
        ("RandomForest_depth3", RandomForestRegressor(max_depth=3, n_estimators=100), 
         {"max_depth": 3, "n_estimators": 100}),
        ("RandomForest_depth5", RandomForestRegressor(max_depth=5, n_estimators=100), 
         {"max_depth": 5, "n_estimators": 100})
    ]
    
    results = []
    
    for model_name, model, params in models:
        with mlflow.start_run(run_name=model_name):
            pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="mean")),
                ("scaler", StandardScaler()),
                ("model", model)
            ])
            
            pipeline.fit(X_train, y_train)
            
            y_pred = pipeline.predict(X_test)
            rmse = np.sqrt(mean_squared_error(y_test, y_pred))
            
            mlflow.log_params(params)
            mlflow.log_metric("rmse", rmse)
            
            mlflow.sklearn.log_model(pipeline, model_name)
            
            results.append((model_name, rmse))
            print(f"{model_name} RMSE: {rmse:.4f}")
    
    return results

model_results = evaluate_models_with_mlflow(
    feature_results['X_train'], 
    feature_results['X_test'], 
    feature_results['y_train'], 
    feature_results['y_test']
)

try:
    experiment_results = mlflow.search_runs(experiment_names=["weather_experiments"])
    print("\nMLflow Experiment Results:")
    print(experiment_results[['run_name', 'metrics.rmse']].head())
except Exception as e:
    print(f"Could not retrieve MLflow results: {e}")

## 7. Conclusion

This notebook demonstrates a complete machine learning pipeline for predicting temperature in London:

1. **Data Loading & Preprocessing**: Loaded weather data and created temporal features
2. **Exploratory Data Analysis**: Visualized temperature trends and feature correlations
3. **Feature Engineering**: Prepared features and target variables with proper handling of missing values
4. **Feature Selection**: Used cross-validation to select optimal features
5. **Model Evaluation**: Compared multiple regression models using MLflow for experiment tracking

The refactored code provides:
- Better organization with clear sections
- Reusable functions for common operations
- Proper error handling and documentation
- Consistent coding style and best practices
- MLflow integration for experiment tracking